In [6]:
import sys
sys.path.append("..")
import src.Whisper_metrics as wm
import evaluate


In [8]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset, Audio

import evaluate

import os
import torch

In [16]:
import os
def load_and_process_data (DATASETS_PATH, FOLDER, FILE_NAME):
    # Charger le dataset
    dataset = load_dataset('csv', data_files=os.path.join("..", "datasets", FOLDER, f"{FILE_NAME}.tsv"), sep="\t")
    PATH_TO_AUDIO = os.path.join(DATASETS_PATH, FOLDER, "audios/") 
    # Creer la colonne audio qui contient le chemin complet + le nom de fichier qui reside dans la colonne path
    dataset["train"] = dataset["train"].map( lambda x: {"audio": PATH_TO_AUDIO + x["audio_file"]})
    """
    cast_columns: change le type d'une colonne
    Audio: permet de transformer un fichier audio en un tableau numpy avece un echantillonage de 16000 khz
    donc la colonne audio contiendra le tableau numpy qui sera apres transformer en spectogramme mel-log
    """
    dataset["train"] = dataset["train"].cast_column("audio", Audio(sampling_rate=16_000))
    return dataset



def whisper_inference (DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME):
    
    
    # get the WordErrorRate module and the CharacterErrorRate module
    wer_metric = evaluate.load("wer", module_type="metric")
    cer_metric = evaluate.load("cer", module_type="metric")


    
    # load model and processor
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    # Recuperer le dataset
    ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)

    # Pour stocker les resultats
    results = []

    # Pour transcire tous les audios
    iterator_ds = iter(ds["train"])

    
    for la_data in iterator_ds:
        # recuperer la colonne audio qui contient le tableau numpy qui represente l'audio
        input_speech = la_data["audio"]
        # transformer le tableau numpy en un spectogramme log-mel
        input_features = processor( input_speech["array"], sampling_rate=input_speech["sampling_rate"], return_tensors="pt").input_features
        # generer les tokens
        predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
        # decoder les tokens (transcire)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        print("label:",la_data["prompt"])
        print(f"predictions:{transcription}")
        
        # stocker les resultats
        results.append({
            "path": la_data["audio_file"], # chemin complet
            "sentence": la_data["prompt"],  # transcription reelle (etiquette)
            "predicted": transcription,  # transcription Whisper        
            "wer": wer_metric.compute(references=[la_data["prompt"]], predictions=[transcription]),   # calcule le      Word Error Rate
            "cer": cer_metric.compute(references=[la_data["prompt"]], predictions=[transcription])    # calcule le Character Error Rate
        })

        wer_metric.add(references=la_data["prompt"], predictions=transcription),   # calcule le      Word Error Rate pour chaque data
        cer_metric.add(references=la_data["prompt"], predictions=transcription)    # calcule le Character Error Rate pour chaque data

    
    return results, wer_metric, cer_metric

In [ ]:
DATASETS_PATH = "/info/raid-etu/m1/s2507404/ASR/projet-m1-asr/datasets/"
LANGUAGE = "en"        
TASK = "transcribe"     

# Pour dev.tsv
FOLDER = "ang/ANG/sps-corpus-1.0-2025-11-25-en"         
FILE_NAME = "ss-corpus-en"        #


results,wer_metric, cer_metric  = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME)


wm.afficher_moyenne_globale_WER_CER(results)
wm.afficher_calcul_WER_CER_post_inference(wer_metric, cer_metric)

label: What was your favorite thing in school?
predictions: My favorite thing in school... Which school though? Oh, it's the library. But that's in my university. In school, like high school, I don't have any favorite thing there. Maybe my friends in high school. But they are not things, aren't they?
label: What was your favorite thing in school?
predictions: I've always enjoyed studying and working with computers in school and out of school.
label: What was your favorite thing in school?
predictions: Probably the social aspect and having friends, but from a subject point of view, my preferred chemistry. And I think that was mainly to do with the actual chemistry teacher who was extremely good. Physics was also quite good, but I wasn't really happy with French.
label: What was your favorite thing in school?
predictions: History
label: What was your favorite thing in school?
predictions: What was your favorite thing in school?
label: How do you get to the doctor?
predictions: I went to 


KeyboardInterrupt



In [20]:
def whisper_inference_v1 (DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME):
    # load model and processor
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    # Recuperer le dataset
    ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)

    # Pour stocker les resultats
    results = []

    # Pour transcire les 10 premiers audio
    for i in range(10):  
        # recuperer la colonne audio qui contient le tableau numpy qui represente l'audio
        input_speech = ds["train"][i]["audio"]
        # transformer le tableau numpy en un spectogramme log-mel
        input_features = processor( input_speech["array"], sampling_rate=input_speech["sampling_rate"], return_tensors="pt").input_features
        # generer les tokens
        predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
        # decoder les tokens (transcire)
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        # stocker les resultats
        results.append({
            "path": ds["train"][i]["audio_file"], # chemin complet
            "sentence": ds["train"][i]["prompt"],  # transcription reelle (etiquette)
            "predicted": transcription  # transcription Whisper
        })
    # affichage
    for r in results:
        print(f"\n {r['path']}")
        print(f"Whisper: {r['predicted']}")
        print(f"label: {r['sentence']}")
        

In [ ]:

DATASETS_PATH = "/info/raid-etu/m1/s2507404/ASR/projet-m1-asr/datasets/"
LANGUAGE = "en"          
TASK = "transcribe"     

# Pour dev.tsv
FOLDER = "ang/ANG/sps-corpus-1.0-2025-11-25-en"          
FILE_NAME = "ss-corpus-en"       



whisper_inference_v1(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME)



 spontaneous-speech-en-63621.mp3
Whisper:  My favorite thing in school... Which school though? Oh, it's the library. But that's in my university. In school, like high school, I don't have any favorite thing there. Maybe my friends in high school. But they are not things, aren't they?
label: What was your favorite thing in school?

 spontaneous-speech-en-67685.mp3
Whisper:  I've always enjoyed studying and working with computers in school and out of school.
label: What was your favorite thing in school?

 spontaneous-speech-en-67740.mp3
Whisper:  Probably the social aspect and having friends, but from a subject point of view, my preferred chemistry. And I think that was mainly to do with the actual chemistry teacher who was extremely good. Physics was also quite good, but I wasn't really happy with French.
label: What was your favorite thing in school?

 spontaneous-speech-en-70984.mp3
Whisper:  History
label: What was your favorite thing in school?

 spontaneous-speech-en-71219.mp3
Wh

In [ ]:
DATASETS_PATH = "/info/raid-etu/m1/s2507404/ASR/projet-m1-asr/datasets/"
LANGUAGE = "en"          
TASK = "transcribe"      

# Pour dev.tsv
FOLDER = "ang/ANG/sps-corpus-1.0-2025-11-25-en"             
FILE_NAME = "ss-corpus-en"        

ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)


In [23]:
input_speech = ds["train"][0]["audio"]

In [25]:
print(input_speech["sampling_rate"])

16000


In [26]:
input_speech["array"]

array([7.5606591e-18, 1.3175864e-17, 1.1452050e-17, ..., 9.0843878e-06,
       8.6455366e-06, 1.2730408e-05], dtype=float32)

In [28]:
 processor = WhisperProcessor.from_pretrained("openai/whisper-small")

In [29]:
  input_features = processor( input_speech["array"], sampling_rate=input_speech["sampling_rate"], return_tensors="pt").input_features

In [30]:
print(input_features)

tensor([[[-0.7899, -0.7899, -0.7899,  ..., -0.7899, -0.7899, -0.7899],
         [-0.7899, -0.7899, -0.7899,  ..., -0.7899, -0.7899, -0.7899],
         [-0.7899, -0.7899, -0.7899,  ..., -0.7899, -0.7899, -0.7899],
         ...,
         [-0.7899, -0.7899, -0.7899,  ..., -0.7899, -0.7899, -0.7899],
         [-0.7899, -0.7899, -0.7899,  ..., -0.7899, -0.7899, -0.7899],
         [-0.7899, -0.7899, -0.7899,  ..., -0.7899, -0.7899, -0.7899]]])


In [31]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")

In [32]:
predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)

In [33]:
print(predicted_ids)

tensor([[50258, 50259, 50359, 50363,  1222,  2954,   551,   294,  1395,   485,
          3013,  1395,  1673,    30,   876,    11,   309,   311,   264,  6405,
            13,   583,   300,   311,   294,   452,  5454,    13,   682,  1395,
            11,   411,  1090,  1395,    11,   286,   500,   380,   362,   604,
          2954,   551,   456,    13,  2704,   452,  1855,   294,  1090,  1395,
            13,   583,   436,   366,   406,   721,    11,  3212,   380,   436,
            30]])


In [34]:
print(len(predicted_ids))
print(len(input_features))

1
1


In [35]:
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
prompt = ds["train"][0]["prompt"]
print(transcription)
print(prompt)

 My favorite thing in school... Which school though? Oh, it's the library. But that's in my university. In school, like high school, I don't have any favorite thing there. Maybe my friends in high school. But they are not things, aren't they?
What was your favorite thing in school?
